Important Features to consider:
1. From https://www.mdpi.com/2072-4292/9/11/1119

NDVI/SAVI/EVI, GLCM, BSI and NDWI

The Normalized Difference Vegetation Index (NDVI) has some drawbacks: for example it is soil dependent at lower values and saturates at higher values. For these reasons other indicators have been proposed, such as the **Soil Adjusted Vegetation Index (SAVI) or Enhanced Vegetation Index (EVI)**. The **Normalized Difference Wetness Index (NDWI)**, has been used either for detecting water or well-watered areas. Because irrigated soils are generally wetter than non-irrigated soils, especially in a semi-arid climate, the NDWI can be a good indicator of irrigation if the observation frequency is sufficient.

References used in this feature creation:

[1] Vegetation Mapping with Random Forest Using Sentinel 2 and GLCM Texture Feature—A Case Study for Lousã Region, Portugal https://www.mdpi.com/2072-4292/14/18/4585

[2] https://www.mdpi.com/2072-4292/9/8/839

[3] Detection of Irrigated Crops from Sentinel-1 and Sentinel-2 Data to Estimate Seasonal Groundwater Use in South India
 https://www.mdpi.com/2072-4292/9/11/1119

 [4] https://docs.digitalearthafrica.org/en/latest/data_specs/Cropland_extent_specs.html

 [5] https://lpdaac.usgs.gov/products/gfsad30afcev001/

 [6] https://www.semanticscholar.org/paper/Nominal-30-m-Cropland-Extent-Map-of-Continental-by-Xiong-Thenkabail/a5295c9910fba51321d08d2b95645a0889ad5bd0

In [ ]:
!pip install geemap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.7 MB/s eta 0:00:00


In [ ]:
import ee
from google.colab import userdata
import geemap

# Set project name
EE_PROJECT_NAME = userdata.get('EE_PROJECT_NAME')

# Authenticate Earth Engine
ee.Authenticate()
# Initialize Earth Engine
ee.Initialize(project=EE_PROJECT_NAME)

# Define Kenya boundary
kenya = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

counties = ee.FeatureCollection('FAO/GAUL/2015/level2') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Pick one county for example
county = counties.filter(ee.Filter.eq('ADM2_NAME', 'Nakuru'))

# Change roi depending on county, state or country
roi = county

In [ ]:
# Print all ADM1 names in Kenya
names = counties.aggregate_array("ADM2_NAME").getInfo()
print(sorted(names))


['Administrative unit not available', 'Administrative unit not available', 'Administrative unit not available', 'Baringo', 'Bomet', 'Bondo', 'Bungoma', 'Buret', 'Busia', 'Butere Mumias', 'Central Kisii', 'Embu', 'Garissa', 'Gucha', 'Homa Bay', 'Ijara', 'Isiolo', 'Kajiado', 'Kakamega', 'Keiyo', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisumu', 'Kitui', 'Koibatek', 'Kuria', 'Kwale', 'Laikipia', 'Lamu', 'Lugari', 'Machakos', 'Makueni', 'Malindi', 'Mandera', 'Maragua', 'Marakwet', 'Marsabit', 'Mbeere', 'Meru Central', 'Meru North', 'Meru South', 'Migori', 'Mombasa', 'Moyale', 'Mt Elgon', 'Muranga', 'Mwingi', 'Nairobi', 'Nakuru', 'Nandi North', 'Nandi South', 'Narok', 'Nyamira', 'Nyandarua', 'Nyando', 'Nyeri', 'Rachuonyo', 'Samburu', 'Siaya', 'Suba', 'Taita Taveta', 'Tana River', 'Teso', 'Tharaka', 'Thika', 'Trans Mara', 'Trans Nzoia', 'Turkana', 'Uasin Gishu', 'Vihiga', 'Wajir', 'West Pokot']


In [ ]:
# JRC Global Surface Water
jrc_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence')
#occurrence is the percentage of times the pixel was classified as water between 1984-2020.
jrc_water_mask = jrc_water.gt(50).clip(roi) #water pixels with more than 50% occurrence

#Other potential datasets (you might need to explore their documentation for appropriate water class values)
# Landsat 8 surface reflectance data
#https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2
# Calculate NDWI for water detection

# Load ESA WorldCover 2020
esa = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map')

# Load ESRI LULC (2020)
#esri = ee.ImageCollection('projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS') \
#    .filterDate('2020-01-01', '2020-12-31') \
#    .first()


# Load GLC_FCS30D for the year 2020
glc = ee.ImageCollection("projects/sat-io/open-datasets/FROM-GLC10").mosaic()


# Load Dynamic World (select most recent composite)
dynamic_world = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
    .filterDate('2020-01-01', '2020-12-31') \
    .select('crops') \
    .mode().clip(roi)


# Load PALSAR Forest/Non-Forest mask
palsar = ee.ImageCollection("JAXA/ALOS/PALSAR/YEARLY/FNF") \
    .filterDate('2017-01-01', '2017-12-31') \
    .first().select('fnf')

# Mask forests: 1 = Forest
forest_mask = palsar.neq(1).clip(roi)

# Water mask from ESA: 80 = Permanent water
water_mask = esa.neq(80).clip(roi)

# Define cropland in each dataset
esa_crop = esa.remap([40], [1]).gt(0).selfMask().clip(roi)  # Cropland class in ESA
glc_crop = glc.remap([10], [1]).gt(0).selfMask().clip(roi)

In [ ]:
# Function to filter pixels with probability > 0.5
def filter_high_probability_pixels(image):
    # Apply a threshold to the probability band
    high_probability_mask = image.gt(0.5)

    # Apply the mask to the image to keep only the pixels with probability > 0.5
    filtered_image = image.updateMask(high_probability_mask)

    return filtered_image


# Apply the function to the probability band.
high_prob_image = filter_high_probability_pixels(dynamic_world.select('crops'))

# Sample the pixels within the ROI using the reducer toList()
high_probability_pixels = high_prob_image.reduceRegion(
    reducer=ee.Reducer.toList(),
    geometry=roi,
    scale=10,  # Adjust the scale as needed
    maxPixels=1e13  # Adjust the maxPixels as needed
)

# Convert high_prob_image to a binary mask (0 and 1)
dw_crop = high_prob_image.gt(0).selfMask().clip(roi)

# Sum agreement
agreement = esa_crop.unmask(0).add(glc_crop.unmask(0)).add(dw_crop.unmask(0)).clip(roi)

# Retain pixels with agreement >= 2
crop_agreement = agreement.gte(3)

# Apply masks
#final_crop = crop_agreement.updateMask(forest_mask.unmask(0)).updateMask(jrc_water_mask.unmask(0))

# Clip to Kenya
#final_crop_kenya = crop_agreement.clip(kenya).selfMask()
final_crop_kenya = crop_agreement.updateMask(crop_agreement).clip(roi)

In [ ]:
# Load the SRTM Digital Elevation Model data
srtm = ee.Image('USGS/SRTMGL1_003')

# Clip the SRTM data to the ROI
srtm_roi = srtm.clip(roi)

In [ ]:
# If it's a mask (boolean), convert it to a 0/1 image
export_image = final_crop_kenya.updateMask(final_crop_kenya).rename('cropland_agreement')
# Export to Google Drive as GeoTIFF
export_task = ee.batch.Export.image.toDrive(
    image=export_image,
    description='final_crop_kenya_export',
    folder='gee_exports',  # Optional: change to your preferred Drive folder
    fileNamePrefix='final_crop_kenya',
    region=kenya.geometry(),  # Exports the image over Kenya's geometry
    scale=10,  # Sentinel-2 resolution
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)

# Start the task
export_task.start()


In [ ]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(kenya, {}, 'Kenya')
Map.centerObject(roi, 8)
#Map.addLayer(esa_crop, {}, 'ESA')
#Map.addLayer(glc_crop, {}, 'FROM-GLC')
#Map.addLayer(high_prob_image, {'min': 0, 'max': 1, 'palette': ['yellow', 'red']}, 'High Probability Pixels')
#Map.addLayer(dw_crop, {}, 'DW')
#Map.addLayer(agreement, {}, 'Agreement')
Map.addLayer(final_crop_kenya, {}, 'Final Crop')
Map.addLayer(roi, {}, 'Nakuru')
#Map.addLayer(jrc_water_mask, {'palette': 'cyan'}, 'JRC Water Mask')
#Map.addLayer(water_mask.lt(1).selfMask(), {}, 'ESA Water Mask')
#Map.addLayer(forest_mask.lt(1).selfMask(), {}, 'Forest Mask')
Map.addLayer(srtm_roi, {}, 'Elevation')
Map

Map(center=[-0.46101926483462224, 36.07982194698872], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
final_crop_nakuru = final_crop_kenya.unmask(0).clip(roi)
Map.addLayer(final_crop_nakuru, {}, 'Crop Nakuru')

In [ ]:
def calculate_bsi(sentinel2_image):
  """
  Calculates the Bare Soil Index (BSI) for a Sentinel-2 image.

  BSI = ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))

  Args:
    sentinel2_image: A Sentinel-2 image (ee.Image). Assumes the image has
                     bands named 'B2', 'B4', 'B8', and 'B11'.

  Returns:
    An ee.Image representing the Bare Soil Index.
  """
  # BSI = ((SWIR1 + Red) - (NIR + Blue)) / ((SWIR1 + Red) + (NIR + Blue))
  bsi = sentinel2_image.expression(
      '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))', {
          'SWIR1': sentinel2_image.select('B11'),
          'RED': sentinel2_image.select('B4'),
          'NIR': sentinel2_image.select('B8'),
          'BLUE': sentinel2_image.select('B2')
      }).rename('BSI')
  return bsi

In [ ]:
# GNDVI, or the Green Normalized Difference Vegetation Index, is a vegetation index used to assess the photosynthetic activity of plants, especially during later stages of development. It's calculated using reflectance data in the green and near-infrared (NIR) bands and is more sensitive to chlorophyll content variations than NDVI. This index is particularly useful for monitoring water and nitrogen uptake in crop canopies and detecting wilted or aging vegetation.
# The MSR (Modified Simple Ratio) index is a vegetation index used in remote sensing to assess vegetation health and biomass. It's calculated using the reflectance values in the near-infrared (NIR) and red bands, similar to the well-known NDVI. MSR is designed to improve linearity and reduce saturation effects compared to simpler indices like NDVI.

#Gaussian normalized difference water index (GNDWI)

# The Red-edge Normalized Difference Vegetation Index (RENDVI) is calculated using the red-edge band (B7) and the red band (B4) of satellite data. It is used to estimate vegetation health and biomass, particularly in the red-edge region where vegetation reflects light strongly. The formula for RENDVI is generally (B7 - B4) / (B7 + B4).

# GLCM texture analysis, using the Gray-Level Co-occurrence Matrix, is a valuable technique for extracting textural information from Sentinel-2 imagery. This approach helps capture the spatial relationships between different gray levels in the image, providing insights into the texture characteristics of different features.


# Function to add indices and GLCM features
def add_all_indices(image):
    # Spectral indices
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B8', 'B11']).rename('NDWI')
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    gndwi = image.normalizedDifference(['B3', 'B11']).rename('GNDWI')
    rendvi = image.normalizedDifference(['B7', 'B4']).rename('RENDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    bsi = image.expression(
        '((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))', {
            'B11': image.select('B11'),
            'B4': image.select('B4'),
            'B8': image.select('B8'),
            'B2': image.select('B2')
        }).rename('BSI')

    msr = image.expression(
        '((NIR / RED - 1) / sqrt(NIR / RED + 1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('MSR')

    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')

    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')

    # Compute NDVI first for GLCM
    image = image.addBands(ndvi)

    # GLCM texture from NDVI (requires 8-bit image)
    ndvi_8bit = ndvi.multiply(255).toUint8()  # Convert to 8-bit for GLCM computation
    glcm = ndvi_8bit.glcmTexture(3)  # Use 3x3 window for texture computation

    # GLCM features
    glcm_asm = glcm.select('NDVI_asm')  # Angular Second Moment
    glcm_contrast = glcm.select('NDVI_contrast')  # Contrast
    glcm_corr = glcm.select('NDVI_corr')  # Correlation
    glcm_var = glcm.select('NDVI_var')  # Variance

    return image.addBands([
        gndwi, ndwi, ndre, rendvi, gndvi, msr, savi, nbr,
        glcm_asm.rename('GLCM_asm'),
        glcm_contrast.rename('GLCM_contrast'),
        glcm_corr.rename('GLCM_corr'),
        glcm_var.rename('GLCM_var')
    ])

In [ ]:
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2020-03-01', '2020-11-01') \
    .filterBounds(kenya) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_all_indices)


# Extract features from Sentinel-2 data for training
training_features = s2.select(
    ['NDVI', 'NDWI', 'NDRE', 'GNDWI', 'RENDVI', 'GNDVI', 'MSR', 'SAVI', 'NBR',
     'GLCM_asm', 'GLCM_contrast', 'GLCM_corr', 'GLCM_var']).mosaic()
# training_features = s2.select(
    # ['NDRE', 'NDWI', 'MSR', 'SAVI', 'NBR',
    #  'GLCM_asm', 'GLCM_contrast', 'GLCM_corr', 'GLCM_var']).mosaic()

# Total number of samples you want to export
n_samples = 50000

# Stratified sample: ensures all classes are represented
sampled_crop_labels = final_crop_kenya.stratifiedSample(
    numPoints=n_samples // 2,
    classBand='crop',  # your target class label
    region=roi.geometry(),
    scale=10,
    seed=42,
    geometries=True  # Include geometry so you can reproject if needed
)

def sample_training_features(training_features, sampled_labels):
    # Get the coordinates of the sampled points.
    sampled_coordinates = sampled_labels.geometry().coordinates()

    # Sample the training data at the sampled point coordinates.
    sampled_training_features = training_features.sampleRegions(
        collection=sampled_labels,
        properties=['crop'],  # Include class labels
        scale=10,  # Adjust as needed
        geometries=True  # Get the geometry of each sampled point
    )

    return sampled_training_features

# Example usage
sampled_crop_training_features = sample_training_features(training_features, sampled_crop_labels)


# Export to Google Drive
# task = ee.batch.Export.table.toDrive(
#     collection=sampled_training_features.filterBounds(county.geometry()),
#     description='training_features',
#     fileFormat='CSV'
# )
# task.start()

# Export to Google Drive
# task = ee.batch.Export.table.toDrive(
#     collection=sampled_labels,
#     description='sampled_class_labels',
#     fileFormat='CSV'
# )
# task.start()


In [ ]:
non_crop_points = final_crop_nakuru.updateMask(final_crop_nakuru.eq(0)).sample(
    region=roi.geometry(),
    scale=10,
    numPixels=n_samples // 2,  # Half the samples for non-crop
    geometries=True
)

# Merge the crop and non-crop samples
sampled_labels = sampled_crop_labels.merge(non_crop_points)

# Add a 'remapped' property to indicate crop (1) and non-crop (0)
#sampled_labels = sampled_labels.map(lambda feature: feature.set('remapped', 1 if feature.get('remapped') == 1 else 0))

# ... (rest of the code) ...

In [ ]:
# Calculate the total number of pixels with value 1
pixel_count = final_crop_kenya.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi.geometry(),  # Use the geometry of your area of interest
    scale=10,  # Adjust the scale if needed
    maxPixels=1e13  # Increase maxPixels if necessary
).get('remapped')  # Get the count for the 'remapped' band

# Get the result as a client-side value
pixel_count_result = pixel_count.getInfo()

print(f"Total pixels with value 1: {pixel_count_result}")

Total pixels with value 1: 217440.0784313725


In [ ]:
def sample_training_features(training_features, sampled_labels):
    # Get the coordinates of the sampled points.
    sampled_coordinates = sampled_labels.geometry().coordinates()

    # Sample the training data at the sampled point coordinates.
    sampled_training_features = training_features.sampleRegions(
        collection=sampled_labels,
        properties=['remapped'],  # Include class labels
        scale=10,  # Adjust as needed
        geometries=True  # Get the geometry of each sampled point
    )

    return sampled_training_features

# Example usage
sampled_training_features = sample_training_features(training_features, sampled_labels)

In [ ]:
sampled_training_features.aggregate_count('remapped').getInfo()

EEException: User memory limit exceeded.

In [ ]:
# Export to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=sampled_training_features.filterBounds(roi.geometry()),
    description='complete_training_features',
    fileFormat='CSV'
)

task.start()

In [ ]:
# Train Random Forest Classifier
classifier = ee.Classifier.smileRandomForest(numberOfTrees=100).train(sampled_training_features, 'remapped')


# Get the feature importance as an Earth Engine dictionary
importance_dict = ee.Dictionary(classifier.explain().get('importance'))
# Export training data to Google Drive

In [ ]:
keys = importance_dict.keys()

# Create a FeatureCollection of feature-importance pairs
def make_feature(key):
    key = ee.String(key)
    value = ee.Number(importance_dict.get(key))
    return ee.Feature(None, {
        'Feature': key,
        'Importance': value
    })

# Create the FeatureCollection
features_fc = ee.FeatureCollection(keys.map(make_feature))

# Sort descending and slice top N
top_n = 10  # change this to however many features you want
sorted_top_features = features_fc.sort('Importance', False).limit(top_n)

# Export to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=sorted_top_features,
    description='Top_10_RF_FeatureImportance',
    fileFormat='CSV'
)
task.start()

In [ ]:
# Sort the features based on importance
sorted_features = ee.List(sorted_top_features.getInfo())

# Select the top 5 most important features
top_5_features = sorted_features.slice(0, 5)

EEException: User memory limit exceeded.

In [ ]:
print(top_5_features.getInfo())
print(sorted_features.getInfo())

In [ ]:

# Create a new image collection with only the top 5 features
selected_features_image = sampled_training_features.select(top_5_features)

# Train a new classifier with the selected features
classifier_with_selected_features = ee.Classifier.smileRandomForest(numberOfTrees=100).train(
    selected_features_image, 'agreement', top_5_features
)

# Apply classifier to entire Kenya using the selected features
classified_image = selected_features_image.classify(classifier_with_selected_features)

# Visualize results
Map = geemap.Map(center=[-1.2921, 36.8219], zoom=6)
Map.addLayer(classified_image, {'min': 0, 'max': 1, 'palette': ['red', 'green']}, 'Cropland Classification')
Map.addLayer(kenya, {}, 'Kenya Boundary')
Map